In [ ]:
import pandas as pd
import anndata as ad
import scanpy as sc
import numpy as np
%run ../../scripts/functions.py
from tensorflow.keras.models import load_model
import scipy.sparse as sp
import sys
#sys.path.append('../../scAAnet')
#from network import ZFixedLayer, DispLayer, DispAct, create_z_fixed, ColwiseMultLayer
sys.path.append('../../scripts')
from modified_network import ZFixedLayer, DispLayer, DispAct, create_z_fixed, ColwiseMultLayer
from sklearn.manifold import MDS
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42


In [ ]:
data_path = '../../data/with_disease_data'
model_path = '../../data/trained_model'
figure_output = '../../figures/exp04/3_figures'

In [ ]:
pathways = pd.read_csv(f'{data_path}/batchfix_paths_with_disease_samples_pathway_counts.csv', sep=',', index_col=0, header=0)
features = pd.read_csv(f'{model_path}/features_model_trained_on.csv').iloc[:, 0]
pathways = pathways[pathways.index.isin(features)]

metadata = pd.read_csv(f'{data_path}/batchfix_paths_with_disease_samples_pathway_metadata.csv', sep=',', index_col=0, header=0)
metadata.rename(columns={'Global.Region': 'Global Region'}, inplace=True)


In [ ]:
concat_ad = convertToAnnData(pathways, metadata)

# load model

In [ ]:
if isinstance(concat_ad, ad.AnnData):
    if sp.issparse(concat_ad.X):
        concat_ad_x = concat_ad.X.todense()
    else:
        concat_ad_x = concat_ad.X
concat_ad_x = np.asmatrix(concat_ad_x).astype('float32')

TPM = concat_ad_x/concat_ad_x.sum(axis=1)
lib_size = concat_ad_x.sum(axis=1)



In [ ]:
model = load_model(
    f'{model_path}/model.h5',
    custom_objects={
        'ZFixedLayer': ZFixedLayer,
        'DispLayer': DispLayer,
        'DispAct': DispAct,
        'create_z_fixed': create_z_fixed
    }
    )
recon = model.predict({'nor_count': TPM, 'lib_size': lib_size})

In [ ]:
encoder = load_model(
    f'{model_path}/trained_encoder.h5',
    custom_objects={
        'ZFixedLayer': ZFixedLayer,
        'DispLayer': DispLayer,
        'DispAct': DispAct,
        'create_z_fixed': create_z_fixed
    }
    )
usage = encoder.predict({'nor_count': TPM, 'lib_size': lib_size})

In [ ]:
decoder = load_model(
    f'{model_path}/trained_decoder.h5',
    custom_objects={
        'ZFixedLayer': ZFixedLayer,
        'DispLayer': DispLayer,
        'DispAct': DispAct,
        'create_z_fixed': create_z_fixed
    }
    )
spectra = decoder.predict(model.get_layer('z_fixed').get_weights()[0])

# AA initial plots

In [ ]:
usage_all = usage
usage_all = pd.DataFrame(usage_all)
usage_all.index = concat_ad.obs.index

usage_all.columns = ['type3', 'type1', 'type2']
desired_order = ['type1', 'type2', 'type3']
usage_all = usage_all[desired_order]
usage_all_meta = pd.concat([usage_all, concat_ad.obs], axis=1)
usage_all_meta.to_csv(f'{data_path}/with_disease_meta_and_usage.csv')
usage_all_meta


In [ ]:
data_at = usage_all.to_numpy()
embedding = MDS(n_components=2, random_state=42)
Y_mds_ats = embedding.fit_transform(spectra)
Y_mds_data = data_at @ Y_mds_ats
(fig, ax) = plt.subplots(1,3,figsize=(25, 8), dpi=600)
plt.suptitle('')
for i in range(1):
    for j in range(3):
        ax[j].set_xticks([])
        ax[j].set_yticks([])
        ax[j].set_xlabel('MDS1', fontsize=15)
        ax[j].set_ylabel('MDS2', fontsize=15)
        g = ax[j].scatter(Y_mds_data[:,0], Y_mds_data[:,1], s=1, alpha=0.5,
                   cmap='inferno', c=usage_all.iloc[:, i*5+j], vmin=0, vmax=1)
        ax[j].set_title('GEP %d' % (i*5+j+1), fontsize=12)
        ax[j].scatter(Y_mds_ats[:,0], Y_mds_ats[:,1], s=200, c='r', zorder=3)
        for k in range(Y_mds_ats.shape[0]):
            ax[j].text(Y_mds_ats[k,0], Y_mds_ats[k,1], k+1, horizontalalignment='center', verticalalignment='center', fontdict={'color': 'white','size':15,'weight':'bold'}, zorder=4)
    plt.colorbar(g, ax=ax[j])
fig.tight_layout()



In [ ]:
concat_ad.obsm['X_umap'] = Y_mds_data
# Compute embedding density for the filtered data
sc.tl.embedding_density(concat_ad, basis='umap', groupby='disease')

# List of disease groups
disease_groups = ['CRC', 'IBD', 'T2D']
K = 3
# Loop through each disease group
for group in disease_groups:
    # Create subplots for healthy and disease densities
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle(f"AA Density Plots: {group} vs Healthy", fontsize=16)

    # Define the study filter based on the current disease group
    study_filter = metadata[metadata.disease == group].study.tolist()

    # Filter the AnnData object
    filtered_ad = concat_ad[concat_ad.obs['study'].isin(study_filter)]

    # Compute embedding density for the filtered data
    sc.tl.embedding_density(filtered_ad, basis='umap', groupby='disease')

    # Extract coordinates and density for the current group and healthy samples
    all_coords = filtered_ad.obsm['X_umap']
    all_density = filtered_ad.obs['umap_density_disease']

    group_coords = filtered_ad[filtered_ad.obs['disease'] == group].obsm['X_umap']
    group_density = filtered_ad[filtered_ad.obs['disease'] == group].obs['umap_density_disease']

    healthy_coords = filtered_ad[filtered_ad.obs['disease'] == 'healthy'].obsm['X_umap']
    healthy_density = filtered_ad[filtered_ad.obs['disease'] == 'healthy'].obs['umap_density_disease']
    print(group, ':' , group_coords.shape[0] + healthy_coords.shape[0])
    # Plot disease density with healthy greyed out
    axes[0].scatter(all_coords[:, 0], all_coords[:, 1], c='lightgray', alpha=0.5, s=10, label='Other')
    scatter_disease = axes[0].scatter(
        group_coords[:, 0], group_coords[:, 1], c=group_density, cmap='Reds', alpha=0.8, s=10, label=group
    )
    for i in range(K):
        axes[0].scatter(Y_mds_ats[i, 0], Y_mds_ats[i, 1], s=500, c='k', zorder=3)
        axes[0].text(
            Y_mds_ats[i, 0], Y_mds_ats[i, 1], str(i + 1),
            horizontalalignment='center', verticalalignment='center',
            fontdict={'color': 'white', 'size': 20, 'weight': 'bold'}, zorder=4
        )
    axes[0].set_title(f"{group} Density", fontsize=14)
    axes[0].set_xlabel("AA 1")
    axes[0].set_ylabel("AA 2")
    plt.colorbar(scatter_disease, ax=axes[0], label="Density")

    # Plot healthy density with disease greyed out
    axes[1].scatter(all_coords[:, 0], all_coords[:, 1], c='lightgray', alpha=0.5, s=10, label='Other')
    scatter_healthy = axes[1].scatter(
        healthy_coords[:, 0], healthy_coords[:, 1], c=healthy_density, cmap='Reds', alpha=0.8, s=10, label='Healthy'
    )
    for i in range(K):
        axes[1].scatter(Y_mds_ats[i, 0], Y_mds_ats[i, 1], s=500, c='k', zorder=3)
        axes[1].text(
            Y_mds_ats[i, 0], Y_mds_ats[i, 1], str(i + 1),
            horizontalalignment='center', verticalalignment='center',
            fontdict={'color': 'white', 'size': 20, 'weight': 'bold'}, zorder=4
        )
    axes[1].set_title(f"Healthy Density ({group} Studies)", fontsize=14)
    axes[1].set_xlabel("AA 1")
    axes[1].set_ylabel("AA 2")
    plt.colorbar(scatter_healthy, ax=axes[1], label="Density")

    # Save the figure
    output_file = f"{figure_output}/figure6A_{group}_vs_healthy_density.pdf"
    plt.tight_layout()
    plt.savefig(output_file, format="pdf")
    plt.show()
